# Test a new layout for nwmd_metrics_by_location_v2

Rebuilds the table into a better physical layout by reading the existing rows, casting the metric
columns, and writing them back. **No metrics are recomputed** — this only moves data that already exists.

Builds a side-by-side test table first so both layouts can be measured before anything replaces the live
one.

## What it changes

| | now | after |
| --- | --- | --- |
| partitioned by | `configuration_name, water_year` | `configuration_name, water_year, quarter` |
| sort order | none | `window_agg, threshold, forecast_lead_time_bin, primary_location_id` |
| point-estimate metrics | `double` | `real` |
| row-group size | default (~76 MB) | 8 MiB |

## Why

Measured on the live table via Trino: one dashboard request returns 5,263 rows (2.37 MB) but reads
**443 MB** in **4.2 s**. `quarter`, `window_agg`, `threshold` and `forecast_lead_time_bin` are always
filtered and prune nothing, so a request scans the whole 3.37M-row partition.

Separately, the 13 point-estimate metrics are `double` and are 878 MB — 58% of the table — while the
bootstrap bounds for the same metrics are already `real`. Matching them shrinks the table ~32%.

## Why rebuild rather than ALTER

- `double` -> `float` is rejected by Iceberg schema evolution (`NOT_SUPPORTED_CHANGE_COLUMN`), so the cast
  needs a rewrite regardless.
- `ADD PARTITION FIELD` leaves the table with two partition specs. Verified side effect: querying the
  `.partitions` metadata table then fails with
  `ValidationException: Cannot find source column for partition field`. A rebuilt table has one clean spec.
- `rewrite_data_files` with `rewrite-all` would rewrite every byte anyway, so a rebuild costs the same.

The sort order and properties are declared on the **empty** table before loading, so rows are sorted as
they are written and no second compaction pass is needed.

## Carrying table properties forward

The API builds its queryables and `ORDER BY` from the `group_by`, `metrics` and `description` table
properties (`teehr-cloud-core/api/src/routes/queryables.py`). A new table does not inherit them, so they
are copied explicitly below. Losing them would make the endpoint reject every filter.

In [ ]:
from teehr.evaluation.spark_session_utils import create_spark_session

TABLE = "nwmd_metrics_by_location_v2"
TEST_TABLE = f"{TABLE}_test"
FQ, TEST_FQ = f"iceberg.teehr.{TABLE}", f"iceberg.teehr.{TEST_TABLE}"

PARTITION_BY = ["configuration_name", "water_year", "quarter"]

# quarter is a partition field now, so it drops out of the sort. What is left
# is the filters nothing else prunes, lowest cardinality first, location last.
SORT_BY = ["window_agg", "threshold", "forecast_lead_time_bin", "primary_location_id"]

# Partitions land near 30MB, so 8MiB gives roughly four row groups each. Lower
# this to skip at a finer grain, at the cost of Parquet metadata and compression.
ROW_GROUP_BYTES = 8 * 1024 * 1024

# Properties the API reads; anything Iceberg manages itself must not be copied.
# "owner" and "provider" are reserved by Spark - setting them raises
# UNSUPPORTED_FEATURE.SET_TABLE_PROPERTY - and the rest are Iceberg's own.
SYSTEM_PROPERTIES = {
    "current-snapshot-id", "format", "format-version", "location", "created-at",
    "sort-order", "owner", "provider",
    "write.parquet.row-group-size-bytes", "write.distribution-mode",
}

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=8,
    executor_memory="16g",
    executor_cores=4,
    aws_profile="default",
)

## Baseline

In [ ]:
def layout(table):
    f = spark.sql(f"""
        SELECT count(*) AS files,
               round(sum(file_size_in_bytes) / 1e6, 1) AS mb,
               round(avg(file_size_in_bytes) / 1e6, 1) AS avg_mb,
               round(avg(cardinality(split_offsets)), 2) AS row_groups_per_file
        FROM {table}.files
    """).collect()[0]
    p = spark.sql(f"""
        SELECT count(*) AS partitions,
               round(avg(total_data_file_size_in_bytes) / 1e6, 1) AS avg_mb,
               round(max(total_data_file_size_in_bytes) / 1e6, 1) AS max_mb
        FROM {table}.partitions
    """).collect()[0]
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {table}").collect()}
    print(f"  {f['files']} files, {f['mb']} MB, {f['avg_mb']} MB avg, "
          f"{f['row_groups_per_file']} row groups/file")
    print(f"  {p['partitions']} partitions, {p['avg_mb']} MB avg, {p['max_mb']} MB max")
    print(f"  sort order: {props.get('sort-order', '(none declared)')}")


ROWS_BEFORE = spark.table(FQ).count()
print(f"{TABLE}: {ROWS_BEFORE:,} rows")
layout(FQ)

## What will be cast

Every `double` becomes `real`. The bootstrap bounds are already `real` and are untouched. float32 carries
~7 significant digits and represents integers exactly to 16,777,216 — ample for metrics rendered to a few
decimals, and for `n_timesteps`. Review the list before continuing.

In [ ]:
columns = [(f.name, f.dataType.simpleString()) for f in spark.table(FQ).schema]
to_cast = [c for c, t in columns if t == "double"]

print(f"{len(columns)} columns; casting {len(to_cast)} double -> float:")
for c in to_cast:
    print(f"    {c}")

copied = {
    r["key"]: r["value"]
    for r in spark.sql(f"SHOW TBLPROPERTIES {FQ}").collect()
    if r["key"] not in SYSTEM_PROPERTIES and not r["key"].startswith("s3.")
}
print("\nproperties carried to the new table:")
for k, v in sorted(copied.items()):
    print(f"    {k} = {v[:90]}")
assert "group_by" in copied, "group_by missing - the API needs it to build queryables"

## Build the test table

Schema and layout first, then load. Declaring the sort order before the insert means rows are sorted as
they are written, so no separate compaction pass is needed.

In [ ]:
select_list = ",\n       ".join(
    f"CAST(`{c}` AS FLOAT) AS `{c}`" if c in to_cast else f"`{c}`"
    for c, _ in columns
)

spark.sql(f"DROP TABLE IF EXISTS {TEST_FQ}")
spark.sql(f"""
    CREATE TABLE {TEST_FQ}
    USING iceberg
    PARTITIONED BY ({", ".join(PARTITION_BY)})
    AS SELECT {select_list}
       FROM {FQ}
       WHERE 1 = 0
""")

order_sql = ", ".join(f"{c} ASC NULLS LAST" for c in SORT_BY)
spark.sql(f"ALTER TABLE {TEST_FQ} WRITE ORDERED BY {order_sql}")

settings = dict(copied)
settings["write.distribution-mode"] = "range"
settings["write.parquet.row-group-size-bytes"] = str(ROW_GROUP_BYTES)
assignments = ", ".join(f"'{k}' = '{v.replace(chr(39), chr(39) * 2)}'"
                        for k, v in settings.items())
spark.sql(f"ALTER TABLE {TEST_FQ} SET TBLPROPERTIES ({assignments})")
print("empty test table created with the target layout")
layout(TEST_FQ)

In [ ]:
%%time
spark.sql(f"INSERT INTO {TEST_FQ} SELECT {select_list} FROM {FQ}")

## Verify nothing changed but the layout

In [ ]:
rows_after = spark.table(TEST_FQ).count()
assert rows_after == ROWS_BEFORE, f"row count differs: {ROWS_BEFORE:,} -> {rows_after:,}"
print(f"row count matches: {rows_after:,}")

# Casting to float32 changes the stored values slightly, so compare at float32
# precision on both sides rather than expecting bit-identical doubles.
probe = "nash_sutcliffe_efficiency"
a = spark.sql(f"SELECT round(sum(CAST({probe} AS FLOAT)), 3) s FROM {FQ}").collect()[0]["s"]
b = spark.sql(f"SELECT round(sum({probe}), 3) s FROM {TEST_FQ}").collect()[0]["s"]
print(f"{probe} checksum  source={a}  rebuilt={b}  match={a == b}")

for key in ("group_by", "metrics", "description"):
    got = {r["key"]: r["value"] for r in
           spark.sql(f"SHOW TBLPROPERTIES {TEST_FQ}").collect()}.get(key)
    print(f"{key}: {'present' if got else 'MISSING - the API needs this'}")

## Compare

In [ ]:
for t in (FQ, TEST_FQ):
    print(t.split(".")[-1])
    layout(t)
    print()

## Measure the real thing

Spark timings are not what the dashboard sees. Port-forward Trino
(`kubectl port-forward -n teehr-hub svc/trino 8080:8080`) and run this against **both** table names,
comparing the `Physical input:` line to the 443 MB / 4.2 s baseline:

```sql
EXPLAIN ANALYZE
SELECT * FROM iceberg.teehr.nwmd_metrics_by_location_v2_test
WHERE primary_location_id LIKE 'usgs-%'
  AND configuration_name = 'nwm30_medium_range'
  AND water_year = '2025'
  AND quarter = '2025-Q1'
  AND window_agg = 'max'
  AND threshold = 'above_q85'
  AND forecast_lead_time_bin = 'P2DT0H_P3DT0H'
ORDER BY primary_location_id, secondary_location_id, configuration_name, unit_name,
         variable_name, member, water_year, quarter, forecast_lead_time_bin,
         threshold, window_agg;
```

## Promote

Only once the numbers above justify it. A rename swap keeps the spec, sort order and properties intact
and leaves the old table recoverable.

**Verify the rename first.** `RENAME TO` is catalog-dependent — it is rejected outright by the Hadoop
catalog (`Cannot rename Hadoop tables`), which is what this was tested against locally. Polaris is a REST
catalog and is expected to support it, but that was not verified here. Try it on a throwaway table in
the namespace before running it on the live one. If the catalog refuses, the fallback is to rebuild v2
from the test table the same way the test table was built — create empty with the layout, set the
properties, then insert.

Anything writing to `nwmd_metrics_by_location_v2` must be idle while this runs.

In [ ]:
# spark.sql(f"ALTER TABLE {FQ} RENAME TO iceberg.teehr.{TABLE}_old")
# spark.sql(f"ALTER TABLE {TEST_FQ} RENAME TO {FQ}")
# layout(FQ)

# Once the dashboard has been verified against the promoted table:
# spark.sql(f"DROP TABLE iceberg.teehr.{TABLE}_old")

In [ ]:
spark.stop()